In [1]:
import os
import time
import requests
import pandas as pd

# ListenBrainz connection
LB_BASE_URL = "https://api.listenbrainz.org/1"
LB_TOKEN = os.getenv("LISTENBRAINZ_TOKEN")  # optional; raises rate limits if set

lb = requests.Session()
lb.headers.update({"User-Agent": "TeamTalentScouts/0.1 (MADS Capstone)"})
if LB_TOKEN:
    lb.headers.update({"Authorization": f"Token {LB_TOKEN}"})


def lb_get(endpoint, params=None, max_retries=3):
    """GET a ListenBrainz endpoint, backing off on rate limits (429)."""
    url = f"{LB_BASE_URL}/{endpoint.lstrip('/')}"
    for _ in range(max_retries):
        resp = lb.get(url, params=params, timeout=30)
        if resp.status_code == 429:
            time.sleep(int(resp.headers.get("X-RateLimit-Reset-In", 5)))
            continue
        resp.raise_for_status()
        return resp
    resp.raise_for_status()

In [2]:
# Test the connection: pull the top 10 sitewide artists (all time)
start = time.time()
resp = lb_get("stats/sitewide/artists", params={"range": "all_time", "count": 10})
elapsed = time.time() - start

print(f"Status: {resp.status_code} | Latency: {elapsed:.2f}s")
print(f"Rate limit remaining: {resp.headers.get('X-RateLimit-Remaining')} / {resp.headers.get('X-RateLimit-Limit')}")

payload = resp.json()["payload"]
top_artists = pd.DataFrame(payload["artists"])
top_artists

Status: 200 | Latency: 0.76s
Rate limit remaining: 29 / 30


,artist_mbid,artist_name,listen_count
0,a74b1b7f-71a5-4011-9441-d0b5e4122711,Radiohead,3624185
1,f59c5520-5f46-4d2c-b2c4-822eabf53419,Linkin Park,3082594
2,056e4f3e-d505-4dad-8ec1-d04f521cbb56,Daft Punk,2969751
3,b10bbbfc-cf9e-42e0-be17-e2c3e1d2600d,The Beatles,2944018
4,381086ea-f511-4aba-bdf9-71c753dc5077,Kendrick Lamar,2853811
5,164f0d73-1234-4e2c-8743-d77bf2191051,Ye,2832351
6,83d91898-7763-47d7-b03b-b92132375c47,Pink Floyd,2629162
7,e21857d5-3256-4547-afb3-4b6ded592596,Gorillaz,2511286
8,cc197bad-dc9c-440d-a5b5-d52ba2e14234,Coldplay,2409583
9,b95ce3ff-3d05-4e87-9e01-c97b66af13d4,Eminem,2406625


In [ ]:
ranges = ("all_time", "year", "this_year")
artist_results = {}

for artist_range in ranges:
    response = lb_get(
        "stats/sitewide/artists",
        params={"range": artist_range, "count": 100},
    )
    artists = response.json()["payload"]["artists"]
    artist_results[artist_range] = pd.DataFrame(artists)
    artist_results[artist_range]["range"] = artist_range
    artist_results[artist_range]["rank"] = range(1, len(artists) + 1)

artist_baseline = artist_results["all_time"].copy()

recent_artists = pd.concat(
    [artist_results["year"], artist_results["this_year"]],
    ignore_index=True,
)
recent_artists = recent_artists.drop_duplicates(subset="artist_mbid")

baseline_mbids = set(artist_baseline["artist_mbid"].dropna())
candidate_seeds = recent_artists[
    ~recent_artists["artist_mbid"].isin(baseline_mbids)
].copy()

artist_baseline.head(), candidate_seeds.head()

Payload keys: ['artists', 'count', 'from_ts', 'last_updated', 'offset', 'range', 'to_ts', 'total_artist_count']
Artist record keys: ['artist_mbid', 'artist_name', 'listen_count']


## Dump feasibility test (Week 1 go/no-go)

Can we build a monthly per-artist listening history (`artist_month`, 2021 onward) from the ListenBrainz full dumps within our time and disk budget?

What we're checking (see `docs/data_pipeline_plan.md` §2b):
1. What the latest full dump contains and how large it is.
2. Whether we can download **only the 2021+ part** of it.
3. What share of listens are mapped to artist MBIDs.
4. How long one file takes to download and aggregate, extrapolated to the whole 2021+ window.

**Approach:** the *Spark* variant of the full dump is an **uncompressed** `.tar` of Parquet files, and the server supports HTTP range requests. So we can:
- index the tar by reading only its 512-byte member headers;
- read each Parquet footer (min/max `listened_at`) to find where 2021 starts;
- download only the columns we need (`listened_at`, `user_id`, `artist_credit_mbids`) from the files in the window, without ever storing the full ~230 GB archive.

In [ ]:
import csv
import io
import re
import tarfile
from pathlib import Path

import duckdb
import numpy as np
import pyarrow.compute as pc
import pyarrow.parquet as pq
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry

DUMP_ROOT = "https://data.metabrainz.org/pub/musicbrainz/listenbrainz/fullexport/"
DATA_START = pd.Timestamp("2021-01-01")
NEEDED_COLS = ["listened_at", "user_id", "artist_credit_mbids"]
CACHE_DIR = Path("data/raw/lb_dumps")
CACHE_DIR.mkdir(parents=True, exist_ok=True)

# Separate session for the dump server: long transfers, so retry dropped connections
dl = requests.Session()
dl.headers.update({"User-Agent": "TeamTalentScouts/0.1 (MADS Capstone)"})
dl.mount("https://", HTTPAdapter(max_retries=Retry(
    total=8, backoff_factor=1, status_forcelist=[429, 500, 502, 503, 504], allowed_methods=None)))

# Latest full dump = highest dump id in the directory listing
listing = dl.get(DUMP_ROOT, timeout=30).text
dump_dirs = sorted(set(re.findall(r'href="(listenbrainz-dump-(\d+)-\d{8}-\d+-full)/"', listing)),
                   key=lambda d: int(d[1]))
DUMP_DIR, DUMP_ID = dump_dirs[-1]
dump_files = re.findall(r'href="([^"]+\.tar(?:\.zst)?)"', dl.get(DUMP_ROOT + DUMP_DIR + "/", timeout=30).text)

sizes = pd.DataFrame(
    [(f, int(dl.head(f"{DUMP_ROOT}{DUMP_DIR}/{f}", timeout=30).headers["Content-Length"]) / 1e9)
     for f in dump_files],
    columns=["file", "size_gb"])
SPARK_URL = f"{DUMP_ROOT}{DUMP_DIR}/" + next(f for f in dump_files if "spark-dump" in f)
print("Latest full dump:", DUMP_DIR)
sizes.round(1)

In [ ]:
class HTTPRangeFile(io.RawIOBase):
    """Seekable read-only view of bytes [start, start + size) of a remote file, fetched with HTTP Range requests."""

    def __init__(self, url, start, size, session=dl):
        self.url, self.start, self.size, self.session = url, start, size, session
        self.pos = 0
        self.bytes_fetched = 0

    def readable(self):
        return True

    def seekable(self):
        return True

    def tell(self):
        return self.pos

    def seek(self, offset, whence=io.SEEK_SET):
        self.pos = {io.SEEK_SET: offset, io.SEEK_CUR: self.pos + offset, io.SEEK_END: self.size + offset}[whence]
        return self.pos

    def readinto(self, buf):
        n = min(len(buf), self.size - self.pos)
        if n <= 0:
            return 0
        first = self.start + self.pos
        resp = self.session.get(self.url, headers={"Range": f"bytes={first}-{first + n - 1}"}, timeout=120)
        resp.raise_for_status()
        buf[:n] = resp.content
        self.pos += n
        self.bytes_fetched += n
        return n


def build_tar_index(url, cache_path):
    """List (name, data_offset, size) of every file in a remote uncompressed tar by walking its headers.

    One ~2 s request per member, so the first run is slow (about an hour); rows are appended as we go
    so a dropped connection resumes where it stopped, and later runs just read the cached CSV.
    """

    def next_header(offset):
        """Return (header offset, TarInfo) of the next real file header, reading a pax header + the
        header after it in a single request."""
        buf = dl.get(url, headers={"Range": f"bytes={offset}-{offset + 2047}"}, timeout=60).content
        pos = 0
        while pos + 512 <= len(buf):
            block = buf[pos:pos + 512]
            if block == b"\0" * 512:  # a zero block marks the end of a tar
                return None, None
            info = tarfile.TarInfo.frombuf(block, "utf-8", "surrogateescape")
            if info.type != tarfile.XHDTYPE:
                return offset + pos, info
            pos += 512 + (info.size + 511) // 512 * 512  # skip the pax header and its data
        return next_header(offset + pos)

    if cache_path.exists() and cache_path.read_text().rstrip().endswith("#complete"):
        return pd.read_csv(cache_path, comment="#")
    offset = 0
    if cache_path.exists():
        done = pd.read_csv(cache_path)
        if len(done):
            last = done.iloc[-1]
            offset = int(last["data_offset"]) + (int(last["size"]) + 511) // 512 * 512
    else:
        cache_path.write_text("name,data_offset,size\n")
    with cache_path.open("a", newline="") as f:
        writer = csv.writer(f)
        while True:
            offset, info = next_header(offset)
            if info is None:
                break
            writer.writerow((info.name.split("/")[-1], offset + 512, info.size))
            f.flush()
            offset += 512 + (info.size + 511) // 512 * 512
        f.write("#complete\n")
    return pd.read_csv(cache_path, comment="#")


tar_index = build_tar_index(SPARK_URL, CACHE_DIR / f"spark_index_{DUMP_ID}.csv")
parquet_index = tar_index[tar_index["name"].str.endswith(".parquet")].copy()
parquet_index["file_no"] = parquet_index["name"].str.removesuffix(".parquet").astype(int)
parquet_index = parquet_index.sort_values("file_no").reset_index(drop=True)
print(f"{len(parquet_index):,} parquet files, {parquet_index['size'].sum() / 1e9:.1f} GB")
parquet_index["size"].div(1e6).describe().round(1).rename("size_mb")

In [ ]:
def open_member(i):
    row = parquet_index.iloc[i]
    return HTTPRangeFile(SPARK_URL, int(row["data_offset"]), int(row["size"]))


def footer_summary(i):
    """Time range, row count and needed-column byte share of one parquet file.

    The dump's parquet files carry no min/max statistics, so the time range comes from reading
    just the listened_at column (~4% of the file); everything else is from the footer.
    """
    pf = pq.ParquetFile(open_member(i))
    meta = pf.metadata
    names = pf.schema_arrow.names
    listened_at = pc.min_max(pf.read(columns=["listened_at"])["listened_at"])
    col_bytes = pd.Series(0, index=names)
    for g in range(meta.num_row_groups):
        rg = meta.row_group(g)
        for c in range(rg.num_columns):
            col_bytes[rg.column(c).path_in_schema.split(".")[0]] += rg.column(c).total_compressed_size
    return {
        "file_no": parquet_index.at[i, "file_no"],
        "min_listened_at": pd.Timestamp(listened_at["min"].as_py()),
        "max_listened_at": pd.Timestamp(listened_at["max"].as_py()),
        "rows": meta.num_rows,
        "needed_col_share": col_bytes[NEEDED_COLS].sum() / col_bytes.sum(),
    }


def first_file_on_or_after(ts):
    """Index of the first parquet file containing listens on/after ts (binary search; files are in time order)."""
    lo, hi = 0, len(parquet_index) - 1
    while lo < hi:
        mid = (lo + hi) // 2
        if footer_summary(mid)["max_listened_at"] < ts:
            lo = mid + 1
        else:
            hi = mid
    return lo


# 1) Are files in time order? Check footers of ~20 evenly spaced files.
sample_ids = sorted(set(np.linspace(0, len(parquet_index) - 1, 20).astype(int)))
footers = pd.DataFrame([footer_summary(i) for i in sample_ids])
print("min_listened_at non-decreasing across sampled files:", footers["min_listened_at"].is_monotonic_increasing)

# 2) First file that contains listens on/after DATA_START
FIRST_IN_WINDOW = first_file_on_or_after(DATA_START)
print(f"First file with listens >= {DATA_START.date()}: index {FIRST_IN_WINDOW}", footer_summary(FIRST_IN_WINDOW))
footers

In [ ]:
window = parquet_index.iloc[FIRST_IN_WINDOW:]
window_footers = footers[footers["file_no"] >= parquet_index.at[FIRST_IN_WINDOW, "file_no"]]
needed_share = window_footers["needed_col_share"].mean()

window_gb = window["size"].sum() / 1e9
print(f"Full spark dump:            {parquet_index['size'].sum() / 1e9:7.1f} GB in {len(parquet_index):,} files")
print(f"2021+ files:                {window_gb:7.1f} GB in {len(window):,} files")
print(f"2021+, only needed columns: {window_gb * needed_share:7.1f} GB  ({needed_share:.0%} of bytes)")

In [ ]:
# Process one typical-size 2021+ file end to end: download needed columns -> artist_month aggregate
SAMPLE_I = int((window["size"] - window["size"].median()).abs().idxmin())

t0 = time.time()
member = open_member(SAMPLE_I)
listens = pq.ParquetFile(member).read(columns=NEEDED_COLS)
download_s = time.time() - t0

t0 = time.time()
con = duckdb.connect()
sample_stats = con.sql("""
    SELECT count(*)                                        AS listens,
           count(*) FILTER (len(artist_credit_mbids) > 0)  AS mapped_listens,
           min(listened_at)                                AS first_listen,
           max(listened_at)                                AS last_listen,
           count(DISTINCT user_id)                         AS users
    FROM listens
""").df()
artist_month_sample = con.sql("""
    WITH artist_listens AS (  -- one row per (listen, credited artist)
        SELECT unnest(artist_credit_mbids)      AS artist_mbid,
               date_trunc('month', listened_at) AS month,
               user_id
        FROM listens
        WHERE len(artist_credit_mbids) > 0
    ),
    per_user AS (
        SELECT artist_mbid, month, user_id, count(*) AS n
        FROM artist_listens
        GROUP BY ALL
    )
    SELECT artist_mbid, month,
           sum(n)                   AS listens,
           count(*)                 AS unique_listeners,
           count(*) FILTER (n >= 2) AS repeat_listeners
    FROM per_user
    GROUP BY ALL
""").df()
aggregate_s = time.time() - t0

sample_stats["mapped_share"] = sample_stats["mapped_listens"] / sample_stats["listens"]
sample_stats["mb_downloaded"] = member.bytes_fetched / 1e6
sample_stats["mb_file_size"] = parquet_index.at[SAMPLE_I, "size"] / 1e6
sample_stats["download_s"] = download_s
sample_stats["aggregate_s"] = aggregate_s
print(f"Sample file {parquet_index.at[SAMPLE_I, 'name']}: {len(artist_month_sample):,} artist-month rows")
sample_stats.T

In [ ]:
# Extrapolate the sample file to the whole 2021+ window (scaled by bytes)
scale = window["size"].sum() / parquet_index.at[SAMPLE_I, "size"]
s = sample_stats.iloc[0]
est = pd.Series({
    "files in window": len(window),
    "est. listens (millions)": s["listens"] * scale / 1e6,
    "est. download, needed columns (GB)": s["mb_downloaded"] * scale / 1e3,
    "measured throughput (MB/s)": s["mb_downloaded"] / s["download_s"],
    "est. download time (h)": s["download_s"] * scale / 3600,
    "est. aggregate time (h)": s["aggregate_s"] * scale / 3600,
    "mapped share of listens": s["mapped_share"],
}).round(2)
est["est. total time, sequential (h)"] = est["est. download time (h)"] + est["est. aggregate time (h)"]
# Each range request waits ~2 s on the server, so downloads are latency-bound: fetching several
# files at once (thread pool) should scale close to linearly until bandwidth becomes the limit
PARALLEL_DOWNLOADS = 8
est[f"est. total time, {PARALLEL_DOWNLOADS} parallel downloads (h)"] = round(
    est["est. download time (h)"] / PARALLEL_DOWNLOADS + est["est. aggregate time (h)"], 2)
est.to_frame("estimate")

### Results (dump 2723, 2026-10-15; run 2026-10-25)

| Check | Result |
|---|---|
| Latest full dump | Listens JSON 249.8 GB, Spark Parquet 248.6 GB (1,512 files of ~180 MB), statistics 24.6 GB |
| Files in time order? | Yes. Each file covers ~3–5 days of listens; 2021-01-01 starts at `700.parquet` |
| 2021+ portion | 812 files, 143.3 GB, but only **~22 GB** for the 3 columns we need (~15% of bytes) |
| Artist MBID mapping | **86%** of listens have `artist_credit_mbids` (sample: 2.15 M listens, Jul 2024) |
| Time per file (sequential) | ~7.7 s download + ~2.5 s DuckDB aggregate |
| Whole 2021+ window | ~1.7 B listens; **~2.2 h sequential, ~45 min with 8 parallel downloads** |
| Disk needed | None for the raw dump (streamed). Output `artist_month` is small |

**Decision: GO.** The dumps are well under the "about a day of compute" threshold, and we never need 230 GB of disk, so the plan's primary history source stands.

**Implications for `ingest/lb_dumps.py`:**
- **Download by HTTP range, not the whole archive.** Use the cached tar index (`data/raw/lb_dumps/spark_index_<dump_id>.csv`), which takes ~1 h to build once per dump.
- **Aggregate per calendar month, not per file.** A month spans ~8–10 files, so unique and repeat listeners can't be summed across files. Download all files of a month, then compute `artist_month` (~20 M rows per month fits easily in DuckDB).
- **Parallelize downloads.** Each range request waits ~2 s on the server, so the work is latency-bound.
- **Footers have no min/max statistics,** so time ranges come from reading `listened_at` (~4% of a file).
- **Report the 14% of unmapped listens** as a known caveat. They're dropped, which may undercount artists whose metadata MusicBrainz doesn't match well.

## Sample feature table (one cohort, as of 2025-12-31)

A small, real version of `processed/features.parquet` (plan §4): **every planned MusicBrainz + ListenBrainz feature** for 5 artists, computed from the actual sources, so the team can see what the modeling input will look like.

- **Artists:**
  - 4 *emerging* artists: first studio release on/after 2023-01-01, and 2025 listeners between 25 and the 1000th-ranked artist's. They're sampled at random from releases LB users listened to in 2023–2025.
  - 1 *baseline* artist (the 2025 sitewide #1), for contrast.
  - To get a larger sample, change `N_EMERGING` / `N_BASELINE`.
- **Point in time:** the as-of date is 2025-12-31. Release, collaboration and label data are filtered to dates ≤ the as-of date, and listening features come from **2025 dump files only**.
- **Sources:**
  - MusicBrainz API (1 req/s): artist, release groups, recordings with credits, releases with labels.
  - ListenBrainz API: 2025 listener stats, similar artists, the sitewide top 1000.
  - The LB Spark dump: one streamed pass over 2025 for the sample artists' monthly listening.
  - Listener counts for collaborators, similar artists and the top 100 use LB's `range=year` stats, which cover the same 2025 window.
- **Output:** `data/processed/sample_features.csv`, plus a column glossary at the end.
- **Not in the sample:**
  - `new_listener_rate` and `months_since_first_listen` need the full 2021+ history, so they come with the full pipeline run.
  - Labels need the 12 months *after* the as-of date, which aren't complete yet for this cohort.
- **Runtime:** ~10–15 minutes on the first run, mostly the 2025 dump pass and the 1 req/s MusicBrainz limit. API responses are cached in `data/raw/api_cache/`, so reruns are much faster.

### Step 1: Setup and API helpers
The MusicBrainz session (1 req/s, User-Agent), ListenBrainz retries, an on-disk cache for every API response, and the `lb_listener_stats` call (ListenBrainz 2025 listeners per artist).

In [ ]:
import hashlib
import json
import random
from concurrent.futures import ThreadPoolExecutor

import pyarrow as pa

AS_OF = pd.Timestamp("2025-12-31")
WINDOW_START = pd.Timestamp("2025-01-01")    # 12-month feature window ends at AS_OF
EMERGING_SINCE = pd.Timestamp("2023-01-01")  # first release within the 3 years before AS_OF
N_EMERGING, N_BASELINE = 4, 1
MIN_LISTENERS = 25
VARIOUS_ARTISTS = "89ad4ac3-39f7-470e-963a-56509c546377"
random.seed(42)

# Hundreds of LB and MB calls below: retry dropped connections and 5xx
# (lb_get already handles 429; mb_get handles MB's 503 rate-limit reply itself)
lb.mount("https://", HTTPAdapter(max_retries=Retry(
    total=5, backoff_factor=1, status_forcelist=[500, 502, 503, 504], allowed_methods=None)))

# MusicBrainz connection (max 1 request/sec; User-Agent must identify the app)
MB_BASE_URL = "https://musicbrainz.org/ws/2"
mb = requests.Session()
mb.headers.update({
    "User-Agent": "TeamTalentScouts/0.1 ( https://github.com/sidmoor/Predicting-Breakout-Success-of-Emerging-Artists )",
    "Accept": "application/json",
})
mb.mount("https://", HTTPAdapter(max_retries=Retry(
    total=5, backoff_factor=2, status_forcelist=[500, 502, 504], allowed_methods=None)))
_mb_last_call = 0.0
API_CACHE = Path("data/raw/api_cache")


def cached_json(key, fetch):
    """Return fetch() and keep it on disk, so reruns skip API calls that already succeeded."""
    path = API_CACHE / f"{key}.json"
    if path.exists():
        return json.loads(path.read_text(encoding="utf-8"))
    data = fetch()
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(data), encoding="utf-8")
    return data


def mb_get(endpoint, params=None, max_retries=3):
    """GET a MusicBrainz ws/2 endpoint (cached), spacing calls 1.1 s apart and retrying on 503 (MB's rate-limit reply)."""
    params = {**(params or {}), "fmt": "json"}

    def fetch():
        global _mb_last_call
        for _ in range(max_retries):
            time.sleep(max(0.0, 1.1 - (time.time() - _mb_last_call)))
            _mb_last_call = time.time()
            resp = mb.get(f"{MB_BASE_URL}/{endpoint}", params=params, timeout=30)
            if resp.status_code == 503:
                continue
            resp.raise_for_status()
            return resp.json()
        resp.raise_for_status()

    key = hashlib.sha1(f"{endpoint}?{sorted(params.items())}".encode()).hexdigest()
    return cached_json(f"mb/{key}", fetch)


def mb_browse(entity, params, max_pages):
    """Page through an MB browse endpoint (100 items per page), stopping after max_pages."""
    items = []
    for page in range(max_pages):
        data = mb_get(entity, {**params, "limit": 100, "offset": page * 100})
        items += data[f"{entity}s"]
        if len(items) >= data[f"{entity}-count"]:
            break
    return items


def mb_date(value):
    """MB partial date ('2023', '2023-05', '2023-05-17') -> first day of that period."""
    if not value:
        return pd.NaT
    return pd.to_datetime(value + "-01" * (2 - value.count("-")), errors="coerce")


def studio_release_groups(release_groups):
    """Dated release groups with no secondary type (drops compilations, live, remixes, soundtracks)."""
    rg = pd.DataFrame({
        "date": [mb_date(r.get("first-release-date")) for r in release_groups],
        "type": [r.get("primary-type") for r in release_groups],
        "secondary": [len(r.get("secondary-types") or []) for r in release_groups],
    })
    return rg[rg["date"].notna() & (rg["secondary"] == 0)]


def lb_listener_stats(mbid):
    """2025 (range=year) listener stats for one artist (cached), or None if LB has none."""
    def fetch():
        resp = lb_get(f"stats/artist/{mbid}/listeners", params={"range": "year"})
        return resp.json()["payload"] if resp.status_code == 200 and resp.content else None

    return cached_json(f"lb_listeners_year/{mbid}", fetch)

### Step 2: Pick the 5 artists (ListenBrainz + MusicBrainz)
Sitewide top 1000 for the listener ceiling, fresh releases for the emerging pool, then a random draw checked against ListenBrainz listener counts and MusicBrainz debut date.

In [ ]:
# 1) Sitewide top 1000 for 2025: the listener ceiling for "emerging", and the set of "top artists"
top_1000 = pd.concat([
    pd.DataFrame(lb_get("stats/sitewide/artists", params={"range": "year", "count": 100, "offset": offset})
                 .json()["payload"]["artists"])
    for offset in range(0, 1000, 100)
], ignore_index=True)
top_1000_mbids = set(top_1000["artist_mbid"].dropna())
ceiling = lb_listener_stats(top_1000["artist_mbid"].dropna().iloc[-1])["total_user_count"]
print(f"Listener ceiling (1000th artist, 2025): {ceiling:,} users")

# 2) Emerging pool: artists on releases from 2023-2025 (fresh-releases looks back 90 days from release_date).
#    Each call is slow (30-60 s), so responses are cached on disk.
def fresh_releases(release_date):
    cache = CACHE_DIR.parent / "lb_api" / f"fresh_releases_{release_date:%Y-%m-%d}.json"
    if not cache.exists():
        cache.parent.mkdir(parents=True, exist_ok=True)
        resp = dl.get(f"{LB_BASE_URL}/explore/fresh-releases/", timeout=180, params={
            "release_date": f"{release_date:%Y-%m-%d}", "days": 90, "past": "true", "future": "false"})
        resp.raise_for_status()
        cache.write_text(resp.text, encoding="utf-8")
    return json.loads(cache.read_text(encoding="utf-8"))["payload"]["releases"]


pool = set()
for release_date in pd.date_range("2023-04-01", "2026-01-01", freq="3MS"):
    for release in fresh_releases(release_date):
        pool.update(release["artist_mbids"] or [])
pool -= top_1000_mbids | {VARIOUS_ARTISTS}
print(f"Emerging pool: {len(pool):,} artists with 2023-2025 releases")

# 3) Draw at random until N_EMERGING pass both checks (LB listeners in range, MB debut on/after EMERGING_SINCE)
release_groups, listener_stats, emerging, checked = {}, {}, [], 0
for mbid in random.sample(sorted(pool), len(pool)):
    if len(emerging) == N_EMERGING:
        break
    checked += 1
    stats = lb_listener_stats(mbid)
    if not stats or not MIN_LISTENERS <= stats["total_user_count"] < ceiling:
        continue
    release_groups[mbid] = mb_browse("release-group", {"artist": mbid}, max_pages=3)
    debut = studio_release_groups(release_groups[mbid])["date"].min()
    if pd.notna(debut) and debut >= EMERGING_SINCE:
        emerging.append(mbid)
        listener_stats[mbid] = stats
print(f"Checked {checked} pool artists to find {len(emerging)} emerging artists")

# 4) Baseline: spread across the 2025 top 100 (just #1 when N_BASELINE = 1)
baseline = top_1000["artist_mbid"].iloc[[0, 20, 40, 60, 80][:N_BASELINE]].tolist()
for mbid in baseline:
    listener_stats[mbid] = lb_listener_stats(mbid)

sample = pd.DataFrame({"artist_mbid": emerging + baseline,
                       "sample_group": ["emerging"] * len(emerging) + ["baseline"] * len(baseline)})
sample

### Step 3: MusicBrainz metadata + ListenBrainz similar artists
The artist (genres/tags, type, country), release groups, recordings with artist credits, and releases with labels.

In [ ]:
# MusicBrainz metadata for the sample (big catalogs are capped: 500 recordings, 300 releases)
SIMILAR_ALGORITHM = "session_based_days_7500_session_300_contribution_5_threshold_10_limit_100_filter_True_skip_30"


def similar_artists(mbid):
    """Top 10 similar artists from LB labs (computed from LB listening sessions), cached."""
    def fetch():
        resp = lb.get("https://labs.api.listenbrainz.org/similar-artists/json",
                      params={"artist_mbids": mbid, "algorithm": SIMILAR_ALGORITHM}, timeout=60)
        return [a["artist_mbid"] for a in resp.json()[:10]] if resp.ok else []
    return cached_json(f"lb_similar/{mbid}", fetch)


t0 = time.time()
mb_data, similar = {}, {}
for n, mbid in enumerate(sample["artist_mbid"], 1):
    mb_data[mbid] = {
        "artist": mb_get(f"artist/{mbid}", {"inc": "genres+tags"}),
        "release_groups": release_groups.get(mbid) or mb_browse("release-group", {"artist": mbid}, max_pages=3),
        "recordings": mb_browse("recording", {"artist": mbid, "inc": "artist-credits"}, max_pages=5),
        "releases": mb_browse("release", {"artist": mbid, "inc": "labels"}, max_pages=3),
    }
    similar[mbid] = similar_artists(mbid)
    print(f"  {n}/{len(sample)} {mb_data[mbid]['artist']['name']}: {len(mb_data[mbid]['recordings'])} recordings, "
          f"{time.time() - t0:.0f}s", flush=True)

# Genres of the 2025 top 100, for the per-genre "top artist" baseline
top_100_genres = {mbid: mb_get(f"artist/{mbid}", {"inc": "genres+tags"})
                  for mbid in top_1000["artist_mbid"].dropna().iloc[:100]}
print(f"MB metadata for {len(mb_data)} sample artists and {len(top_100_genres)} top-100 artists, {time.time() - t0:.0f}s")

### Step 4: ListenBrainz dump: monthly listening for 2025
Streams the 2025 dump files and counts listens and unique and repeat listeners per artist per month, plus platform-wide active users.

In [ ]:
# One streamed pass over the 2025 dump files for the sample artists' monthly listening + platform totals.
# (Collaborator / similar / top-100 listener counts come from the LB API's 2025 stats instead; keeping
# per-user rows for ~1000 artists across a whole year doesn't fit in memory. The full pipeline avoids
# this by aggregating one month at a time.)
allowlist = set(sample["artist_mbid"])
allow_table = pa.table({"artist_mbid": sorted(allowlist)})

window_end = AS_OF + pd.Timedelta(days=1)
file_ids = range(first_file_on_or_after(WINDOW_START), first_file_on_or_after(window_end) + 1)
in_window = f"listened_at >= TIMESTAMP '{WINDOW_START}' AND listened_at < TIMESTAMP '{window_end}'"
print(f"Scanning {len(file_ids)} files for {len(allowlist):,} artists ...")


SCAN_DIR = Path(f"data/interim/sample_scan_{DUMP_ID}_{AS_OF:%Y%m%d}")
SCAN_DIR.mkdir(parents=True, exist_ok=True)


def scan_file(i):
    """Per-file (month, user) listen counts for everyone, and (month, user, artist) counts for the allowlist.

    Results are saved per file, so a crash or rerun picks up where it stopped. Each file's ~2M listens
    are freed as soon as it's aggregated, keeping memory flat across the pass.
    """
    platform_path, artist_path = SCAN_DIR / f"{i}_platform.parquet", SCAN_DIR / f"{i}_artist.parquet"
    if not (platform_path.exists() and artist_path.exists()):
        listens = pq.ParquetFile(open_member(i)).read(columns=NEEDED_COLS)
        con = duckdb.connect()
        try:
            con.register("listens", listens)
            con.register("allow", allow_table)
            pq.write_table(con.sql(f"""
                SELECT date_trunc('month', listened_at) AS month, user_id, count(*) AS listens
                FROM listens WHERE {in_window} GROUP BY ALL
            """).fetch_arrow_table(), platform_path)
            pq.write_table(con.sql(f"""
                SELECT month, user_id, artist_mbid, count(*) AS listens
                FROM (SELECT date_trunc('month', listened_at) AS month, user_id, unnest(artist_credit_mbids) AS artist_mbid
                      FROM listens WHERE {in_window} AND len(artist_credit_mbids) > 0)
                WHERE artist_mbid IN (SELECT artist_mbid FROM allow)
                GROUP BY ALL
            """).fetch_arrow_table(), artist_path)
        finally:
            con.close()
            del listens
    return pq.read_table(platform_path), pq.read_table(artist_path)


t0 = time.time()
platform_parts, artist_parts = [], []
with ThreadPoolExecutor(max_workers=4) as pool_exec:
    for n, (platform, artist) in enumerate(pool_exec.map(scan_file, file_ids), 1):
        platform_parts.append(platform)
        artist_parts.append(artist)
        if n % 20 == 0:
            print(f"  {n}/{len(file_ids)} files, {time.time() - t0:.0f}s")

# Months span several files, so re-aggregate per (month, user) before counting unique/repeat listeners
con = duckdb.connect()
con.register("platform_rows", pa.concat_tables(platform_parts))
con.register("artist_rows", pa.concat_tables(artist_parts))
con.execute("CREATE TABLE artist_user_month AS SELECT artist_mbid, month, user_id, sum(listens) AS listens FROM artist_rows GROUP BY ALL")
platform_month = con.sql("SELECT month, count(DISTINCT user_id) AS active_users, sum(listens) AS listens FROM platform_rows GROUP BY month ORDER BY month").df()
platform_users_12m = con.sql("SELECT count(DISTINCT user_id) FROM platform_rows").fetchone()[0]
artist_month = con.sql("""
    SELECT artist_mbid, month, sum(listens) AS listens, count(*) AS unique_listeners,
           count(*) FILTER (listens >= 2) AS repeat_listeners
    FROM artist_user_month GROUP BY ALL
""").df()
artist_12m = con.sql("""
    SELECT artist_mbid, sum(listens) AS listens_12m, count(*) AS unique_listeners_12m,
           count(*) FILTER (listens >= 2) AS repeat_listeners_12m
    FROM (SELECT artist_mbid, user_id, sum(listens) AS listens FROM artist_user_month GROUP BY ALL)
    GROUP BY artist_mbid
""").df().set_index("artist_mbid")
print(f"Done in {time.time() - t0:.0f}s: {len(artist_month):,} artist-month rows, {platform_users_12m:,} active LB users in 2025")
platform_month

### Step 5: Turn the raw data into features: `sample_features.csv`

In [ ]:
PARENT_GENRES = {  # MB genre -> ~15 parent genres; first keyword match wins, so specific groups come first
    "hip-hop": ["hip hop", "rap", "trap", "drill", "grime"],
    "r&b/soul": ["r&b", "soul", "funk"],
    "metal": ["metal", "deathcore"],
    "punk/emo": ["punk", "hardcore", "emo"],
    "electronic": ["electronic", "house", "techno", "edm", "dubstep", "drum and bass", "trance", "ambient",
                   "synthwave", "hyperpop", "idm", "electro", "breakbeat"],
    "latin": ["latin", "reggaeton", "cumbia", "salsa", "bachata", "corrido", "regional mexican"],
    "k-pop/j-pop": ["k-pop", "j-pop", "j-rock", "city pop", "anime"],
    "reggae/afro": ["reggae", "dancehall", "afrobeat", "amapiano"],
    "country": ["country", "americana", "bluegrass"],
    "folk": ["folk", "singer-songwriter", "acoustic"],
    "jazz/blues": ["jazz", "blues", "swing"],
    "classical/soundtrack": ["classical", "orchestral", "soundtrack", "opera", "baroque"],
    "pop": ["pop", "dance"],
    "rock": ["rock", "indie", "alternative", "grunge", "shoegaze"],
}
MAJOR_LABEL_KEYWORDS = ["universal", "sony", "warner", "columbia", "rca", "epic", "atlantic", "interscope",
                        "republic", "capitol", "def jam", "island", "geffen", "motown", "polydor", "parlophone",
                        "elektra", "virgin", "emi", "arista", "300 entertainment", "aftermath"]


def mb_genres(artist):
    """MB genres by vote count, falling back to tags when an artist has no genres."""
    tags = artist.get("genres") or artist.get("tags") or []
    return [t["name"] for t in sorted(tags, key=lambda t: -t.get("count", 0))]


def parent_genre(genres):
    for genre in genres:
        for parent, keywords in PARENT_GENRES.items():
            if any(k in genre for k in keywords):
                return parent
    return None


def label_class(label_names, artist_name):
    names = [n.lower() for n in label_names]
    if any(k in n for n in names for k in MAJOR_LABEL_KEYWORDS):
        return "major"
    if names and all(n in ("[no label]", artist_name.lower()) for n in names):
        return "self-released"
    return "independent" if names else "unknown"


def listening_trend(mbid):
    """Relative 6-month slope and 3-month mean month-over-month growth of monthly unique listeners."""
    months = pd.date_range(WINDOW_START, AS_OF, freq="MS")
    monthly = (artist_month[artist_month["artist_mbid"] == mbid].set_index("month")["unique_listeners"]
               .reindex(months, fill_value=0))
    last6 = monthly.iloc[-6:].to_numpy(dtype=float)
    slope_6m = np.polyfit(np.arange(6), last6, 1)[0] / max(last6.mean(), 1)
    mom = monthly.iloc[-4:].to_numpy(dtype=float)
    mom_growth_3m = np.mean(np.diff(mom) / np.maximum(mom[:-1], 1))
    return slope_6m, mom_growth_3m


def listeners_12m(mbids):
    """2025 unique listeners: dump aggregate for sample artists, LB API (range=year, the same 2025 window) for others."""
    return pd.Series({
        m: artist_12m.at[m, "unique_listeners_12m"] if m in artist_12m.index
        else (lb_listener_stats(m) or {}).get("total_user_count", 0)
        for m in mbids
    }, dtype=float)


rows = []
for mbid, group in sample.itertuples(index=False):
    d, stats = mb_data[mbid], listener_stats[mbid] or {}
    artist = d["artist"]
    genres = mb_genres(artist)

    rg = studio_release_groups(d["release_groups"])
    rg = rg[rg["date"] <= AS_OF]
    dates, types = rg["date"].sort_values(), rg["type"].value_counts()

    recs = [r for r in d["recordings"] if pd.notna(mb_date(r.get("first-release-date")))
            and mb_date(r.get("first-release-date")) <= AS_OF]
    rec_collabs = [{c["artist"]["id"] for c in r.get("artist-credit", [])} - {mbid, VARIOUS_ARTISTS} for r in recs]
    collab_set = set().union(*rec_collabs) if rec_collabs else set()
    collab_listeners = listeners_12m(collab_set)

    labels = {li["label"]["name"] for rel in d["releases"]
              if pd.notna(mb_date(rel.get("date"))) and mb_date(rel.get("date")) <= AS_OF
              for li in rel.get("label-info", []) if li.get("label")}

    l12 = artist_12m.reindex([mbid]).iloc[0].fillna(0)
    top10 = sum(l["listen_count"] for l in stats.get("listeners", [])[:10])
    slope_6m, mom_growth_3m = listening_trend(mbid)

    rows.append({
        "artist_mbid": mbid, "artist_name": artist["name"], "sample_group": group, "as_of_date": AS_OF.date(),
        # Release activity
        "first_release_date": dates.min().date() if len(dates) else None,
        "career_age_days": (AS_OF - dates.min()).days if len(dates) else np.nan,
        "n_release_groups": len(rg),
        "n_albums": types.get("Album", 0), "n_eps": types.get("EP", 0), "n_singles": types.get("Single", 0),
        "releases_last_12m": int((dates >= WINDOW_START).sum()),
        "median_days_between_releases": dates.diff().dt.days.median(),
        "days_since_last_release": (AS_OF - dates.max()).days if len(dates) else np.nan,
        "album_to_single_ratio": types.get("Album", 0) / max(types.get("Single", 0), 1),
        # Listening (2025 dump)
        "listens_12m": int(l12["listens_12m"]),
        "unique_listeners_12m": int(l12["unique_listeners_12m"]),
        "listens_per_listener": l12["listens_12m"] / max(l12["unique_listeners_12m"], 1),
        "repeat_listener_ratio": l12["repeat_listeners_12m"] / max(l12["unique_listeners_12m"], 1),
        "listener_share_12m": l12["unique_listeners_12m"] / platform_users_12m,
        "listener_slope_6m": slope_6m,
        "mom_growth_3m": mom_growth_3m,
        "top10_listener_share": top10 / stats["total_listen_count"] if stats.get("total_listen_count") else np.nan,
        # Genre
        "mb_genres": "; ".join(genres[:3]),
        "primary_genre": parent_genre(genres),
        "n_genres": len(artist.get("genres") or []),
        # Collaboration network
        "n_recordings": len(recs),
        "n_unique_collaborators": len(collab_set),
        "pct_recordings_with_features": np.mean([len(c) > 0 for c in rec_collabs]) if recs else np.nan,
        "max_collaborator_listeners_12m": collab_listeners.max() if collab_set else 0,
        "mean_collaborator_listeners_12m": collab_listeners.mean() if collab_set else 0,
        "collaborated_with_top_artist": bool(collab_set & top_1000_mbids),
        "similar_artists_mean_listeners_12m": listeners_12m(similar[mbid]).mean() if similar[mbid] else np.nan,
        # Region / label
        "artist_type": artist.get("type"),
        "country": artist.get("country"),
        "area": (artist.get("area") or {}).get("name"),
        "n_labels": len(labels),
        "has_label": bool(labels - {"[no label]"}),
        "label_class": label_class(labels, artist["name"]),
    })
features = pd.DataFrame(rows)

# Genre-relative features: vs. the 2025 top 100 in the same parent genre, and percentile within the sample
top_100 = pd.DataFrame({"artist_mbid": list(top_100_genres),
                        "primary_genre": [parent_genre(mb_genres(a)) for a in top_100_genres.values()]})
top_100["unique_listeners_12m"] = listeners_12m(top_100["artist_mbid"]).to_numpy()
genre_top_median = top_100.groupby("primary_genre")["unique_listeners_12m"].median()
features["genre_baseline_gap"] = np.log10(
    features["unique_listeners_12m"].clip(lower=1) / features["primary_genre"].map(genre_top_median))
features["listener_percentile_within_genre"] = (
    features.groupby("primary_genre")["unique_listeners_12m"].rank(pct=True))

out_path = Path("data/processed/sample_features.csv")
out_path.parent.mkdir(parents=True, exist_ok=True)
features.round(4).to_csv(out_path, index=False)
print(f"Saved {features.shape[0]} artists x {features.shape[1]} columns -> {out_path}")
features

### Column glossary (`sample_features.csv`)

All features are as of `as_of_date`. "12m" means the 12 months ending on it (2025 for this sample).

| Group | Column | Definition | Source |
|---|---|---|---|
| ID | `artist_mbid`, `artist_name` | MusicBrainz artist ID and name | MB |
| ID | `sample_group` | `emerging` (debut ≥ 2023, below the top-1000 ceiling) or `baseline` (2025 sitewide top) | LB + MB |
| Release | `first_release_date`, `career_age_days` | Earliest studio release group; days from it to the as-of date | MB release groups |
| Release | `n_release_groups`, `n_albums`, `n_eps`, `n_singles` | Studio release groups by type, dated ≤ as-of (no compilations, live or remixes) | MB |
| Release | `releases_last_12m` | Release groups in the 12m window | MB |
| Release | `median_days_between_releases`, `days_since_last_release` | Release cadence and recency | MB |
| Release | `album_to_single_ratio` | Albums ÷ max(singles, 1) | MB |
| Listening | `listens_12m`, `unique_listeners_12m` | Listens and distinct LB users in the 12m window | LB dump |
| Listening | `listens_per_listener` | Listens ÷ unique listeners (depth of engagement) | LB dump |
| Listening | `repeat_listener_ratio` | Share of 12m listeners with ≥ 2 listens | LB dump |
| Listening | `listener_share_12m` | Unique listeners ÷ all active LB users in the window (adjusts for LB's own growth) | LB dump |
| Listening | `listener_slope_6m` | Slope of monthly unique listeners over the last 6 months ÷ their mean | LB dump |
| Listening | `mom_growth_3m` | Mean month-over-month listener growth over the last 3 months | LB dump |
| Listening | `top10_listener_share` | Share of 12m listens from the artist's top 10 listeners (superfan concentration) | LB API (`range=year`) |
| Genre | `mb_genres`, `n_genres` | Top 3 MB genres (tags if none) and the number of MB genres | MB |
| Genre | `primary_genre` | First MB genre mapped to one of ~15 parent genres (`PARENT_GENRES`) | MB |
| Genre | `genre_baseline_gap` | log10(unique listeners ÷ median of the 2025 top-100 artists in the same parent genre) | LB dump + MB |
| Genre | `listener_percentile_within_genre` | Percentile of unique listeners within the parent genre (here: within the sample) | LB dump |
| Collaboration | `n_recordings`, `n_unique_collaborators` | Recordings dated ≤ as-of; distinct co-credited artists | MB recordings |
| Collaboration | `pct_recordings_with_features` | Share of recordings with ≥ 1 co-credited artist | MB |
| Collaboration | `max_/mean_collaborator_listeners_12m` | 12m unique listeners of the collaborators | MB + LB dump |
| Collaboration | `collaborated_with_top_artist` | Any collaborator in the 2025 sitewide top 1000 | MB + LB API |
| Collaboration | `similar_artists_mean_listeners_12m` | Mean 12m listeners of the 10 most similar artists | LB labs + LB dump |
| Region/label | `artist_type`, `country`, `area` | Person/Group, ISO country, MB area | MB |
| Region/label | `n_labels`, `has_label`, `label_class` | Labels on releases ≤ as-of; major / independent / self-released / unknown (keyword list `MAJOR_LABEL_KEYWORDS`) | MB releases |

**Known simplifications in this sample:**
- Large catalogs are capped (500 recordings, 300 releases, 300 release groups), which affects some `baseline` artists' counts.
- MB genres, LB similar-artist lists and `top10_listener_share` are current snapshots, not reconstructed as of the date. That's fine for 2025, but in the historical cohorts they'd leak future information, so they need care.
- With 5 artists, `listener_percentile_within_genre` is computed within the sample only (mostly 1.0). It becomes meaningful on the full candidate pool.

## Raw pull (before any feature engineering)
The MusicBrainz and ListenBrainz data behind the sample, flattened to one CSV per source table in `data/sample_raw/`. Nothing is filtered to the as-of date or turned into features here, so a few release dates after 2025-12-31 still appear. Listen counts from the dump are summed to month level, because the dump has ~2M individual listens per file. LB usernames are left out.

In [ ]:
# Raw pull for the sample: the API/dump fields as returned, flattened to one CSV per source table.
# No feature engineering, filtering by as-of date, or genre mapping. That all happens in the feature cell above.
RAW_DIR = Path("data/sample_raw")
SCAN_DIR = Path(f"data/interim/sample_scan_{DUMP_ID}_{AS_OF:%Y%m%d}")  # per-file results of the 2025 dump pass
RAW_DIR.mkdir(parents=True, exist_ok=True)
names = dict(zip(sample["artist_mbid"], [mb_data[m]["artist"]["name"] for m in sample["artist_mbid"]]))


def joined(items, key="name"):
    return "; ".join(str(i[key]) for i in items or [])


def counted(items):
    return "; ".join(f"{i['name']}:{i.get('count', '')}" for i in items or [])


def save(df, name):
    df.to_csv(RAW_DIR / f"{name}.csv", index=False, encoding="utf-8-sig")
    print(f"  {name}.csv: {len(df):,} rows x {df.shape[1]} cols")


# --- MusicBrainz ---
artist_rows = []
for group, lookups in [("sample", {m: mb_data[m]["artist"] for m in sample["artist_mbid"]}),
                       ("top100_reference", top_100_genres)]:
    for mbid, a in lookups.items():
        life = a.get("life-span") or {}
        artist_rows.append({
            "artist_mbid": mbid, "name": a.get("name"), "row_group": group,
            "sample_group": sample.set_index("artist_mbid")["sample_group"].get(mbid),
            "type": a.get("type"), "gender": a.get("gender"), "country": a.get("country"),
            "area": (a.get("area") or {}).get("name"), "begin_area": (a.get("begin-area") or {}).get("name"),
            "life_span_begin": life.get("begin"), "life_span_end": life.get("end"), "ended": life.get("ended"),
            "disambiguation": a.get("disambiguation"),
            "genres": counted(a.get("genres")), "tags": counted(a.get("tags")),
        })
save(pd.DataFrame(artist_rows), "mb_artists")

save(pd.DataFrame([
    {"artist_mbid": m, "artist_name": names[m], "release_group_mbid": r["id"], "title": r.get("title"),
     "primary_type": r.get("primary-type"), "secondary_types": "; ".join(r.get("secondary-types") or []),
     "first_release_date": r.get("first-release-date")}
    for m in sample["artist_mbid"] for r in mb_data[m]["release_groups"]
]), "mb_release_groups")

save(pd.DataFrame([
    {"artist_mbid": m, "artist_name": names[m], "recording_mbid": r["id"], "title": r.get("title"),
     "length_ms": r.get("length"), "first_release_date": r.get("first-release-date"), "video": r.get("video"),
     "credited_artists": joined(r.get("artist-credit")),
     "credited_artist_mbids": "; ".join(c["artist"]["id"] for c in r.get("artist-credit") or []),
     "join_phrases": "|".join(c.get("joinphrase", "") for c in r.get("artist-credit") or [])}
    for m in sample["artist_mbid"] for r in mb_data[m]["recordings"]
]), "mb_recordings")

save(pd.DataFrame([
    {"artist_mbid": m, "artist_name": names[m], "release_mbid": r["id"], "title": r.get("title"),
     "date": r.get("date"), "country": r.get("country"), "status": r.get("status"),
     "label_mbid": (li.get("label") or {}).get("id"), "label_name": (li.get("label") or {}).get("name"),
     "label_type": (li.get("label") or {}).get("type"), "catalog_number": li.get("catalog-number")}
    for m in sample["artist_mbid"] for r in mb_data[m]["releases"] for li in (r.get("label-info") or [{}])
]), "mb_releases_labels")

# --- ListenBrainz API ---
# Every artist whose 2025 listener stats were used: the sample, their collaborators, similar artists, the top 100
roles = {}  # first role wins: sample > collaborator > similar > top100
for mbid in top_1000["artist_mbid"].dropna().iloc[:100]:
    roles[mbid] = "top100"
for m in sample["artist_mbid"]:
    roles.update({s: "similar" for s in similar[m] or []})
for m in sample["artist_mbid"]:
    roles.update({c["artist"]["id"]: "collaborator" for r in mb_data[m]["recordings"] for c in r.get("artist-credit") or []})
roles.update({m: "sample" for m in names})
stats_rows, top_listener_rows = [], []
for path in sorted((API_CACHE / "lb_listeners_year").glob("*.json")):
    p = json.loads(path.read_text(encoding="utf-8"))
    if not p or p["artist_mbid"] not in roles:
        continue
    in_sample = p["artist_mbid"] in names
    stats_rows.append({
        "artist_mbid": p["artist_mbid"], "artist_name": p.get("artist_name"), "role": roles[p["artist_mbid"]],
        "range": p.get("range"), "from": pd.to_datetime(p["from_ts"], unit="s").date(),
        "to": pd.to_datetime(p["to_ts"], unit="s").date(),
        "total_listen_count": p.get("total_listen_count"), "total_user_count": p.get("total_user_count"),
        "last_updated": pd.to_datetime(p["last_updated"], unit="s"),
    })
    if in_sample:  # LB usernames left out: only the per-listener counts feed top10_listener_share
        top_listener_rows += [{"artist_mbid": p["artist_mbid"], "artist_name": p.get("artist_name"),
                               "listener_rank": rank, "listen_count": li["listen_count"]}
                              for rank, li in enumerate(p.get("listeners") or [], 1)]
save(pd.DataFrame(stats_rows), "lb_artist_listener_stats_2025")
save(pd.DataFrame(top_listener_rows), "lb_top_listeners_2025")

save(pd.DataFrame([
    {"artist_mbid": m, "artist_name": names[m], "similar_rank": rank, "similar_artist_mbid": s}
    for m in sample["artist_mbid"] for rank, s in enumerate(similar[m] or [], 1)
]), "lb_similar_artists")

save(top_1000.assign(rank=range(1, len(top_1000) + 1)), "lb_sitewide_top1000_2025")

# --- ListenBrainz dump (2025 files): per-file counts from the scan, summed to month level ---
con = duckdb.connect()
scan = SCAN_DIR.as_posix()
lb_artist_month = con.sql(f"""
    SELECT artist_mbid, month::DATE AS month, sum(listens) AS listens, count(*) AS unique_listeners
    FROM (SELECT artist_mbid, month, user_id, sum(listens) AS listens
          FROM read_parquet('{scan}/*_artist.parquet') GROUP BY ALL)
    GROUP BY ALL ORDER BY artist_mbid, month
""").df()
lb_artist_month.insert(1, "artist_name", lb_artist_month["artist_mbid"].map(names))
save(lb_artist_month, "lb_dump_artist_month_2025")
save(con.sql(f"""
    SELECT month::DATE AS month, count(DISTINCT user_id) AS active_users, sum(listens) AS listens
    FROM read_parquet('{scan}/*_platform.parquet') GROUP BY month ORDER BY month
""").df(), "lb_dump_platform_month_2025")
con.close()